Trash below

In [ ]:
import pyaudio, wave, datetime, keyboard, whisper, os
from scipy.io import wavfile
import numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from gtts import gTTS
from TTS.api import TTS
import torch

d:\Projects\Python\minor\col\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Use Pyaudio to record audio and save it as a WAV file
FORMAT = pyaudio.paInt16
CHANNELS = 1
RATE = 16000
CHUNK = 1024


# Function to generate a timestamped filename
def get_filename():
    time_stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    return time_stamp

# Function to record audio until 'q' is pressed and save it as a WAV file
def record_audio():
    p = pyaudio.PyAudio()
    stream = p.open(format=FORMAT, channels=CHANNELS, rate=RATE, input=True, frames_per_buffer=CHUNK)
    frames = []
    print("Recording ... Press 'q' to stop.")

    while True:
        data = stream.read(CHUNK)
        frames.append(data)
        
        if keyboard.is_pressed('q'):  # If 'q' is pressed, stop recording
            print("Recording stopped.")
            break

    stream.stop_stream()
    stream.close()
    p.terminate()

    FILENAME = get_filename() + ".wav"

    wf = wave.open(FILENAME, 'wb')
    wf.setnchannels(CHANNELS)
    wf.setsampwidth(p.get_sample_size(FORMAT))
    wf.setframerate(RATE)
    wf.writeframes(b''.join(frames))
    wf.close()
    print(f"Audio saved as {FILENAME}")
    return FILENAME



In [3]:
filename = record_audio()

Recording ... Press 'q' to stop.
Recording stopped.
Audio saved as 20251031_185342.wav


In [4]:
# aud = wave.open(filename, "rb")
# read the wav file and assign data as ndarray
sample_rate, data = wavfile.read(filename)
type(data)
# convert to float32 if not already in that format
if data.dtype != np.float32:
    data = data.astype(np.float32) / np.iinfo(data.dtype).max

In [5]:
# openai whisper model for stt
model = whisper.load_model("tiny")

d:\Projects\Python\minor\col\Lib\site-packages\torch\_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [6]:
# model expects np.ndarray as input in float32 format
result = model.transcribe(data)
print('\n')
# print transcription
print("Transcription:", result["text"])

text = result["text"]

d:\Projects\Python\minor\col\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")




Transcription:  So, I think this audio is being recorded.


In [ ]:
# model_name = "Helsinki-NLP/opus-mt-en-hi"
# tokenizer = MarianTokenizer.from_pretrained(model_name)
# translator = MarianMTModel.from_pretrained(model_name)

d:\Projects\Python\minor\col\Lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
d:\Projects\Python\minor\col\Lib\site-packages\transformers\models\marian\tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [ ]:
# def translate_text(text, model, tokenizer):
#     inputs = tokenizer([text], return_tensors="pt", padding=True)
#     translated = model.generate(**inputs)
#     output = tokenizer.decode(translated[0], skip_special_tokens=True)
#     return output

In [ ]:
# english_text = result["text"]
# hindi_text = translate_text(english_text, translator, tokenizer)
# print("Hindi Translation:", hindi_text)

Hindi Translation: तो, मुझे लगता है कि यह ऑडियो रिकॉर्ड किया जा रहा है लगता है.


In [ ]:
translator = pipeline("translation",
                      model = "facebook/nllb-200-distilled-600M",
                      torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                      device=0 if torch.cuda.is_available() else -1
                      )

NLLB_CODES = {
    "en": "eng_Latn",
    "hi": "hin_Deva",
    "de": "deu_Latn",
    "ja": "jpn_Jpan",
}

translated = translator(
    text, "ja", "hi"
)[0]["translation_text"]

print(translated)

In [10]:
def text_to_speech(text, lang='hi'):
    tts = gTTS(text=text, lang=lang)
    tts.save(f"{filename[:-4]}output.mp3")

    print(f"Saved translated audio as {filename[:-4]}output.mp3")
    if os.name == "nt":  # Windows
        os.startfile(f"{filename[:-4]}output.mp3")
    else:  # macOS/Linux
        os.system(f"mpg123 {filename}output")

    
    # os.system("start output.mp3")  # For Windows, use "afplay output.mp3" for MacOS or "xdg-open output.mp3" for Linux

In [11]:
# hindi_text = "नमस्ते, आप कैसे हैं?"
text_to_speech(hindi_text)

Saved translated audio as 20251031_185342output.mp3


In [12]:
import torch
# Some models pickle objects that reference project classes (e.g. BaseDatasetConfig).
# Add those classes to torch's safe globals before unpickling so loading succeeds.
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import XttsAudioConfig, XttsArgs
# The unpickler complained about TTS.config.shared_configs.BaseDatasetConfig
from TTS.config.shared_configs import BaseDatasetConfig


# Allowlist known classes used inside the saved model during unpickling
# torch.serialization.safe_globals([
#     XttsConfig,
#     XttsAudioConfig,
#     BaseDatasetConfig,
#     XttsArgs
# ])

# # Load the TTS model inside the safe_globals context as an extra precaution
# with torch.serialization.safe_globals([XttsConfig, XttsAudioConfig, BaseDatasetConfig, XttsArgs]):
tts = TTS(model_name="tts_models/multilingual/multi-dataset/xtts_v2")


# torch.serialization.

 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
 > Using model: xtts


In [14]:


# Load a multi-speaker model
# tts = TTS(model_name="tts_models/multilingual/multi-dataset/xtts_v2")

# User's reference voice
reference_wav = "20251031_185342.wav"

# Your translated text
# hindi_text = "नमस्ते, आप कैसे हैं?"

# Generate Hindi speech in user's voice
tts.tts_to_file(text=hindi_text, speaker_wav=reference_wav, file_path="cloned_hindi.wav", language="hi",)
print("Voice-cloned audio saved as cloned_hindi.wav")


 > Text splitted to sentences.
['तो, मुझे लगता है कि यह ऑडियो रिकॉर्ड किया जा रहा है लगता है.']
 > Processing time: 19.52897024154663
 > Real-time factor: 3.7541305780626932
Voice-cloned audio saved as cloned_hindi.wav
